In [8]:
import re
import jieba
import jieba.posseg as pseg
from pygments.lexer import words
from sudachipy.sudachipy import SplitMode
from yaml import tokens

try:
    from sudachipy import tokenizer as sudachi_tokenizer
    from sudachipy import dictionary
    SUDACHI_AVAILABLE = True
    sudachi_dict = dictionary.Dictionary().create()
    sudachi_mode = sudachi_tokenizer.Tokenizer.SplitMode.C
except ImportError:
    SUDACHI_AVAILABLE = False
    print("Warning: Sudachi not available. Install with: pip install sudachipy sudachidict-core")

try:
    from konlpy.tag import Okt
    # Test if Java is actually available
    try:
        # source ~/.zshrc or ~/.bashrc to ensure JAVA_HOME is set, if needed
        okt = Okt()
        okt.morphs("테스트")
        KONLPY_AVAILABLE = True
    except:
        KONLPY_AVAILABLE = False
        print("Warning: KoNLPy installed but Java not available. Using fallback for Korean.")
except ImportError:
    KONLPY_AVAILABLE = False

try:
    from pythainlp.tokenize import word_tokenize as thai_tokenize
    PYTHAINLP_AVAILABLE = True
except ImportError:
    PYTHAINLP_AVAILABLE = False

In [2]:
# load ecklectic data
import pandas as pd
df = pd.read_csv("/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/data/eclektic/raw/eclektic_7.csv")

# generate a new column 'eclektic_id' by concatenating 'language' string,'q_id' int64
df[f"eclektic_id"] = df["language"] + "_" + df["q_id"].astype(str)

df.head()

,original_lang,original_content,original_question,original_answer,content,question,answer,language,translated,q_id,title,url,eclektic_id
0,zh,武安侯是一个曾经在多个王朝使用的封侯名，可以指以下人物：赵兴（先秦）：公元前222年，秦国灭...,西汉册封谁为武安侯？,田蚡,Marquis of Wu'an is a feudal title used in sev...,Who was conferred the title of Marquis of Wu'a...,Tian Fen,en,1,502,武安侯,https://zh.wikipedia.org/wiki/武安侯,en_502
1,zh,武安侯是一个曾经在多个王朝使用的封侯名，可以指以下人物：赵兴（先秦）：公元前222年，秦国灭...,西汉册封谁为武安侯？,田蚡,Le titre de Marquis de Wu'an a été utilisé dan...,Qui a été fait Marquis de Wu'an sous la dynast...,Tian Fen,fr,1,502,武安侯,https://zh.wikipedia.org/wiki/武安侯,fr_502
2,zh,武安侯是一个曾经在多个王朝使用的封侯名，可以指以下人物：赵兴（先秦）：公元前222年，秦国灭...,西汉册封谁为武安侯？,田蚡,ווֹאָן הוֹאוּ הוא תואר אצולה שהיה בשימוש במספר...,"מי קיבל את התואר ""המרקיז של ווֹאָן"" בתקופת שוש...",טְייֵן פֿוּן,he,1,502,武安侯,https://zh.wikipedia.org/wiki/武安侯,he_502
3,zh,武安侯是一个曾经在多个王朝使用的封侯名，可以指以下人物：赵兴（先秦）：公元前222年，秦国灭...,西汉册封谁为武安侯？,田蚡,武安侯是一个曾经在多个王朝使用的封侯名，可以指以下人物：赵兴（先秦）：公元前222年，秦国灭...,西汉册封谁为武安侯？,田蚡,zh,0,502,武安侯,https://zh.wikipedia.org/wiki/武安侯,zh_502
4,zh,武安侯是一个曾经在多个王朝使用的封侯名，可以指以下人物：赵兴（先秦）：公元前222年，秦国灭...,西汉册封谁为武安侯？,田蚡,"무안후는 여러 왕조에서 사용되었던 작위명으로, 다음 인물을 지칭할 수 있습니다.\n...",서한은 누구를 무안후로 봉했습니까?,전분,ko,1,502,武安侯,https://zh.wikipedia.org/wiki/武安侯,ko_502


In [ ]:
filtered_tokens_dict = {}


# HEBREW KEYWORDS


In [4]:
he_df = df[df["language"] == "he"]
he_df.shape

(216, 13)

In [ ]:
from transformers import BertModel, BertTokenizerFast

alephbert_tokenizer = BertTokenizerFast.from_pretrained('onlplab/alephbert-base')
alephbert = BertModel.from_pretrained('onlplab/alephbert-base')

# if not finetuning - disable dropout
alephbert.eval()


In [ ]:
he_keep_pos = {
  "NOUN",   # common nouns
  "PROPN",  # proper nouns / names (matches zh: nr/ns/nt and NER-like usefulness)
  "VERB",   # verbs
  "ADJ",    # adjectives
  "ADV",    # adverbs
  "NUM"    # numbers (years, counts)
}

from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('dicta-il/dictabert-morph')
model = AutoModel.from_pretrained('dicta-il/dictabert-morph', trust_remote_code=True)

model.eval()

# iterate each row in df, tokenize the 'question' column, and keep only tokens with POS in he_keep_pos



for index, row in he_df.iterrows():
    sentence = row['question']
    result = model.predict([sentence], tokenizer)
    # keep only tokens with POS in he_keep_pos
    filtered_tokens = [token for token in result[0].get("tokens", []) if token['pos'] in he_keep_pos]
    eclektic_id = row[f"eclektic_id"]
    # save it accumulated in a dictionary with key as eclektic_id and value as filtered_tokens list
    filtered_tokens_dict[eclektic_id] = {"question": sentence, "tokens": filtered_tokens}




/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# JA POS

In [6]:
ja_df = df[df["language"] == "ja"]
ja_df.shape

(216, 13)

In [ ]:
ja_non_functional_pos = [
"名詞",
"動詞",
"形容詞",
"副詞",
]
tokenizer_obj = dictionary.Dictionary().create()
for index, row in ja_df.iterrows():
    sentence = row['question']
    m = tokenizer_obj.tokenize(sentence, SplitMode.C)
    # filter to non-functional POS
    for m in m:
        if m.part_of_speech()[0] in ja_non_functional_pos:
            eclektic_id = row[f"eclektic_id"]
            if eclektic_id not in filtered_tokens_dict:
                filtered_tokens_dict[eclektic_id] = {"question": sentence, "tokens": []}
            filtered_tokens_dict[eclektic_id]["tokens"].append({"token": m.surface(), "pos": m.part_of_speech()})
            



In [8]:
# get the number of unique eclektic_ids we have in the filtered_tokens_dict
num_unique_eclektic_ids = len(filtered_tokens_dict)
print(f"Number of unique eclektic_ids with filtered tokens: {num_unique_eclektic_ids}")

Number of unique eclektic_ids with filtered tokens: 432


# KO POS

In [9]:
ko_df = df[df["language"] == "ko"]
ko_df.shape

(216, 13)

In [ ]:
preserve_pos = [
"Noun",
"Verb",
"Adjective",
"Adverb",
"Number",
"Foreign",
"Alpha"
]

okt = Okt()

for index, row in ko_df.iterrows():
    sentence = row['question']
    # tokens = okt.morphs(sentence)
    pos_tags = okt.pos(sentence)
    for token, pos in pos_tags:
        if pos in preserve_pos:
            eclektic_id = row[f"eclektic_id"]
            if eclektic_id not in filtered_tokens_dict:
                filtered_tokens_dict[eclektic_id] = {"question": sentence, "tokens": []}
            filtered_tokens_dict[eclektic_id]["tokens"].append({"token": token, "pos": pos})





In [14]:
# get the number of unique eclektic_ids we have in the filtered_tokens_dict
num_unique_eclektic_ids = len(filtered_tokens_dict)
print(f"Number of unique eclektic_ids with filtered tokens: {num_unique_eclektic_ids}")

Number of unique eclektic_ids with filtered tokens: 648


# ZH POS

In [15]:
zh_df = df[df["language"] == "zh"]
zh_df.shape

(216, 13)

In [10]:
zh_non_functional_pos = [
    "n","f","s","t",
    "nr","ns","nt","nw","nz",
    "v","vd","vn",
    "a","ad","an",
    "d",
    "m","q","eng",
 
    "PER","LOC","ORG","TIME"
]

In [15]:
sentence = "光绪二十九年对应西元哪一年？"
words = pseg.cut(sentence, use_paddle=True)  # if you add vi mapping
print("Using jieba.posseg with Paddle:")
print(words)
for tok in words:
    # show word,flag only if it's in our non-functional POS list
    if tok.flag in zh_non_functional_pos:   
        print(f"{tok.word} ({tok.flag})")

Using jieba.posseg with Paddle:
<generator object cut at 0x16f286840>
光绪 (n)
二十九年 (m)
对应 (vn)
西 (f)
元 (m)


In [6]:
zh_non_functional_pos = [
    "n","f","s","t",
    "nr","ns","nt","nw","nz",
    "v","vd","vn",
    "a","ad","an",
    "d",
    "m","q","eng",
    "PER","LOC","ORG","TIME"
]

for index, row in zh_df.iterrows():
    sentence = row['question']
    words = pseg.cut(sentence, use_paddle=True)  # if you add vi mapping
    for tok in words:
        # show word,flag only if it's in our non-functional POS list
        if tok.flag in zh_non_functional_pos:   
            eclektic_id = row[f"eclektic_id"]
            if eclektic_id not in filtered_tokens_dict:
                filtered_tokens_dict[eclektic_id] = {"question": sentence, "tokens": []}
            filtered_tokens_dict[eclektic_id]["tokens"].append({"token": tok.word, "pos": tok.flag})



NameError: name 'zh_df' is not defined

In [7]:
# get the number of unique eclektic_ids we have in the filtered_tokens_dict
num_unique_eclektic_ids = len(filtered_tokens_dict)
print(f"Number of unique eclektic_ids with filtered tokens: {num_unique_eclektic_ids}")

NameError: name 'filtered_tokens_dict' is not defined

In [18]:
# save the filtered_tokens_dict to a json file
import json
with open("/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/src/wimbd_cooc_features/pos_keywords/filtered_pos_tokens_dict.json", "w") as f:
    json.dump(filtered_tokens_dict, f, ensure_ascii=False, indent=4)

# for Eng, Fr, Es POS

In [1]:
import stanza

langs = ["en", "fr", "es"]

pipelines = {
    lang: stanza.Pipeline(lang, processors="tokenize,pos", use_gpu=False)
    for lang in langs
}

text = "Paris est la capitale de la France."

doc = pipelines["fr"](text)

for sent in doc.sentences:
    for word in sent.words:
        print(word.text, word.upos)

/Users/anniewang/Desktop/MLLM-interpretability/MLLM-interpretability/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-07 14:55:36 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-03-07 14:55:37 INFO: Downloaded file to /Users/anniewang/stanza_resources/resources.json
2026-03-07 14:55:37 WARNING: Language en package default expects mwt, which has been added
2026-03-07 14:55:37 INFO: Loading these models for language: en (English):
| Processor | Package         |
-------------------------------
| tokenize  | combined        |
| mwt       | combined        |
| pos       | combined_charlm |

2026-03-07 14:55:37 INFO: Using device: cpu
2026-03-07 1

Paris PROPN
est AUX
la DET
capitale NOUN
de ADP
la DET
France PROPN
. PUNCT


In [ ]:
import stanza

nlp = stanza.Pipeline("zh", processors="tokenize,ner")

text = "光绪二十九年对应西元哪一年？"

doc = nlp(text)

for ent in doc.ents:
    print(ent.text, ent.type)

2026-03-07 15:02:19 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-03-07 15:02:19 INFO: Downloaded file to /Users/anniewang/stanza_resources/resources.json
2026-03-07 15:02:19 INFO: "zh" is an alias for "zh-hans"
2026-03-07 15:02:20 INFO: Loading these models for language: zh-hans (Simplified_Chinese):
| Processor | Package   |
-------------------------
| tokenize  | gsdsimp   |
| ner       | ontonotes |

2026-03-07 15:02:20 INFO: Using device: cpu
2026-03-07 15:02:20 INFO: Loading: tokenize
2026-03-07 15:02:20 INFO: Loading: ner
2026-03-07 15:02:23 INFO: Done loading processors!


二十九年 DATE
